In [ ]:
# =============================================================================
# NOTEBOOK 1: Pure Parametric Baseline (Cubic Splines)
# Architecture: Mathematical Interpolation (No Machine Learning)
# Description: Fits a flexible geometric curve to known option quotes to fill
#              in the missing gaps. Ensures a smooth volatility smile but 
#              does not capture intraday microstructure or bid-ask noise.
# =============================================================================

import re
import warnings
import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

warnings.filterwarnings("ignore")

INPUT_CSV = "../../data/raw/dataset.csv"

SAMPLE_SUB_CSV = "../../data/raw/sandbox_solution.csv"
KAGGLE_OUT     = "../../submission_phase1_spline.csv"

IV_CLIP_LO     = 0.02  # Absolute minimum volatility
IV_CLIP_HI     = 1.50  # Absolute maximum volatility to prevent arbitrage tails


In [ ]:

# ── 2. DATA LOADING ───────────────────────────────────────────────────────────
print("Step 1: Loading Option Chain Data...")
df = pd.read_csv(INPUT_CSV, parse_dates=["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)

# Isolate the Implied Volatility columns from the standard features
iv_cols = [c for c in df.columns if c not in ("datetime", "underlying_price")]

# Split into Call Options (CE) and Put Options (PE)
ce_cols = sorted([c for c in iv_cols if c.endswith("CE")])
pe_cols = sorted([c for c in iv_cols if c.endswith("PE")])

print(f"  Total Rows: {len(df)} | CE Strikes: {len(ce_cols)} | PE Strikes: {len(pe_cols)}")
print(f"  Total Missing NaNs to fill: {df[iv_cols].isna().sum().sum()}")


In [ ]:

# ── 3. STRIKE PRICE EXTRACTION ────────────────────────────────────────────────
def get_strike(col: str) -> int:
    """Extracts the numeric strike price from the column string (e.g., NIFTY...24000CE -> 24000)"""
    m = re.search(r"(\d+)(?:CE|PE)$", col)
    return int(m.group(1)) if m else 0

strike_map = {c: get_strike(c) for c in iv_cols}

# ── 4. THE MATHEMATICAL ENGINE (CUBIC SPLINES) ────────────────────────────────
def fit_spline_surface(df_work: pd.DataFrame, cols: list) -> pd.DataFrame:
    """
    Loops through every single minute (row) in the dataset, looks at the strikes 
    we actually have data for, and draws a smooth mathematical curve through them 
    to guess the missing NaNs.
    """
    df_out = df_work.copy()
    strikes = np.array([strike_map[c] for c in cols])
    
    for idx, row in df_work.iterrows():
        # Extract the true market IVs for this specific minute
        iv_values = row[cols].values.astype(float)
        
        valid_mask = ~np.isnan(iv_values) & (iv_values > 0)
        
        # A Cubic Spline requires a minimum of 4 points to calculate curvature
        if valid_mask.sum() >= 4:
            valid_strikes = strikes[valid_mask]
            valid_ivs = iv_values[valid_mask]
            
            # FIT THE MATH: Draw the flexible geometric curve
            # bc_type='natural' forces the ends of the curve to be straight lines,
            # preventing the model from predicting extreme/impossible loops in the deep OTM tails.
            cs = CubicSpline(valid_strikes, valid_ivs, bc_type='natural', extrapolate=True)
            
            # PREDICT THE GAPS: 
            missing_mask = ~valid_mask
            if missing_mask.any():
                predicted_ivs = cs(strikes[missing_mask])
                
                # SAFETY SHIELD: Do not allow the math to predict negative volatility
                predicted_ivs = np.clip(predicted_ivs, IV_CLIP_LO, IV_CLIP_HI)
                
                # Inject the predictions back into the dataframe
                df_out.loc[idx, np.array(cols)[missing_mask]] = predicted_ivs
                
    return df_out

print("\nStep 2: Fitting Cubic Splines for Call Options (CE)...")
df = fit_spline_surface(df, ce_cols)

print("Step 3: Fitting Cubic Splines for Put Options (PE)...")
df = fit_spline_surface(df, pe_cols)


In [ ]:

# ── 5. FINAL SAFETY CLEANUP ───────────────────────────────────────────────────
# If any row had fewer than 4 data points, the Spline skipped it. 
# We use a simple Forward-Fill -> Backward-Fill to patch any surviving NaNs.
remaining = df[iv_cols].isna().sum().sum()
if remaining > 0:
    print(f"\nStep 4: Patching {remaining} surviving un-splinable NaNs with Time-Series FFill...")
    df[iv_cols] = df[iv_cols].ffill().bfill()
else:
    print("\nStep 4: Surface is 100% complete. No surviving NaNs.")


In [ ]:

# ── 6. KAGGLE SUBMISSION FORMATTING ───────────────────────────────────────────
print("\nStep 5: Formatting for Kaggle Private Leaderboard...")

# 1. Melt the wide surface matrix into a long list
df_long = df.melt(
    id_vars=['datetime'], 
    value_vars=iv_cols,
    var_name='contract', 
    value_name='predicted_iv'
)

# 2. Prevent the American Date Collision Bug
if not pd.api.types.is_datetime64_any_dtype(df_long['datetime']):
    df_long['datetime'] = pd.to_datetime(df_long['datetime'], format='%d-%m-%Y %H:%M')

# 3. Create the strict Kaggle target string: "datetime||contract"
df_long['datetime_str'] = df_long['datetime'].dt.strftime('%d-%m-%Y %H:%M')
df_long['id'] = df_long['datetime_str'] + '||' + df_long['contract']

# 4. Join securely against the hidden targets
try:
    sub_template = pd.read_csv(SAMPLE_SUB_CSV)
    
    # Left merge ensures we only submit the exact rows Kaggle wants (drops the training data)
    final_sub = pd.merge(
        sub_template[['id']], 
        df_long[['id', 'predicted_iv']], 
        on='id', 
        how='left'
    )
    
    final_sub = final_sub.rename(columns={'predicted_iv': 'value'})
    
    # Ultimate Failsafe for missing joins
    missing_joins = final_sub['value'].isna().sum()
    if missing_joins > 0:
        print(f"  WARNING: {missing_joins} IDs failed to join. Falling back to median.")
        final_sub['value'] = final_sub['value'].fillna(0.20)
    
    # KAGGLE UPGRADE: Exact 6-Decimal Rounding
    final_sub['value'] = final_sub['value'].round(6)
    
    # Export
    final_sub.to_csv(KAGGLE_OUT, index=False)
    print(f"\nSUCCESS! Phase 1 Parametric Baseline saved to: {KAGGLE_OUT}")
    print(f"Total Submitted Rows: {len(final_sub)} | Expected: {len(sub_template)}")

except FileNotFoundError:
    print(f"\nCRITICAL ERROR: Could not find '{SAMPLE_SUB_CSV}'. Please ensure it is in the same directory.")